# Xiexie wake-word — openWakeWord training notebook

Trains a custom **"Xiexie"** wake-word detector and exports `xiexie.onnx` to your
Google Drive. Designed to run end-to-end on a vanilla **Colab T4** (free tier).

**Pipeline** (faithful to the official openWakeWord auto-trainer):

1. Pip-install `openwakeword`, the `piper-sample-generator` fork, and the training extras.
2. Download MIT room-impulse-responses + an AudioSet shard + 1 hour of FMA music
   for background noise / reverb augmentation.
3. Download `~2,000 hours` of pre-computed openWakeWord features (ACAV100M) plus an
   `~11 h` validation set — provides the negative class without raw audio.
4. Generate ~N synthetic positives (`"Xiexie"`, varied speaker/speed/noise) using the
   multi-speaker `en_US-libritts_r-medium` Piper model.
5. Generate adversarial negatives (phoneme-overlapping phrases) automatically.
6. **Mix in the user's local recordings** from `positives.zip` (highest-quality positives, oversampled).
7. Augment all positives with RIRs + background noise.
8. Train the small DNN classifier on top of the frozen Google speech-embedding backbone.
9. Evaluate (recall on held-out positives, false-trigger rate on a noise slice).
10. Copy `xiexie.onnx` to `MyDrive/xiexie/xiexie.onnx`.

**Total time on T4:** ~3–5 h end-to-end with the default config; ~10 h on a CPU-only
Colab session. Tune the `CONFIG` cell below if you want a quicker smoke run.

**Heads-up on Colab session limits:** free-tier sessions disconnect after ~12 h of
uptime and ~90 min of inactivity. Keep this tab focused; if you get bumped you can
rerun the notebook — every step skips work that's already on disk.


## 0 — Configuration

Tweak everything from this single cell. Defaults are tuned for a 3–5 h T4 run.


In [ ]:
CONFIG = {
    # --- core ---
    "wake_word": "Xiexie",
    "model_name": "xiexie",

    # --- dataset sizes ---
    # synthetic positives generated by Piper (the libritts multi-speaker model
    # gives accent/prosody diversity for free).
    "n_samples_train": 2000,
    "n_samples_val": 500,
    # how many copies of each user recording we duplicate into positive_train
    # (real recordings are scarce + high-signal so we oversample them).
    "user_recording_duplication": 8,

    # --- training ---
    "steps": 15000,
    "layer_size": 32,
    "target_accuracy": 0.7,
    "target_recall": 0.5,
    "target_false_positives_per_hour": 0.2,
    "max_negative_weight": 1500,

    # --- background data (more = more robust, but more disk + time) ---
    # AudioSet shard `bal_train09.tar` is ~1.5 GB; keep it for default runs.
    "audioset_shard": "bal_train09.tar",
    "fma_hours": 1,

    # --- where to put the final model on Drive ---
    "drive_output_dir": "/content/drive/MyDrive/xiexie",
    # path inside Drive where you'll upload the recorded positives zip.
    # if missing the run still works (synthetic-only positives).
    "drive_positives_zip": "/content/drive/MyDrive/xiexie/positives.zip",
}
CONFIG


## 1 — Install dependencies

Mirrors the official `automatic_model_training.ipynb`. Pinned versions are what the
openWakeWord maintainer tested against — Colab moves fast, do not bump these casually.


In [ ]:
# piper-sample-generator (Rhasspy fork) — TTS pipeline used by openwakeword's auto-trainer.
!git clone https://github.com/rhasspy/piper-sample-generator || true
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install -q piper-phonemize
!pip install -q webrtcvad

# openwakeword (full install — needed for train.py).
!git clone https://github.com/dscripka/openwakeword || true
!pip install -q -e ./openwakeword

# Pinned training deps (matches official auto-train notebook).
!pip install -q mutagen==1.47.0 \
                torchinfo==1.8.0 \
                torchmetrics==1.2.0 \
                speechbrain==0.5.14 \
                audiomentations==0.33.0 \
                torch-audiomentations==0.11.0 \
                acoustics==0.2.6 \
                tensorflow-cpu==2.8.1 \
                tensorflow_probability==0.16.0 \
                onnx_tf==1.10.0 \
                pronouncing==0.2.0 \
                datasets==2.14.6 \
                deep-phonemizer==0.0.19

# openwakeword bundled resources (mel-spec + Google speech embedding) — needed by augment + inference.
import os
os.makedirs('./openwakeword/openwakeword/resources/models', exist_ok=True)
for fname in [
    'embedding_model.onnx', 'embedding_model.tflite',
    'melspectrogram.onnx', 'melspectrogram.tflite',
]:
    !wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{fname} \
        -O ./openwakeword/openwakeword/resources/models/{fname}

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(),
      'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Standard imports used across the rest of the notebook.
import os, sys, shutil, uuid, glob, json, time
from pathlib import Path
import numpy as np
import scipy, scipy.io.wavfile
import yaml
import datasets
from tqdm.auto import tqdm


## 2 — Mount Drive and stage user-recorded positives

Run `python -m record_samples` on your Mac first (see `training/wakeword/README.md`),
zip the resulting `positives/` folder, and upload `positives.zip` to **`MyDrive/xiexie/`**
before continuing.

If the zip is missing the notebook still trains a synthetic-only model — quality drops
noticeably (no real-mic, no real-room, no your-voice positives) but the rest of the
pipeline works.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

USER_POSITIVES_DIR = Path('/content/user_positives')
USER_POSITIVES_DIR.mkdir(exist_ok=True)

zip_path = Path(CONFIG['drive_positives_zip'])
if zip_path.exists():
    !unzip -q -o {zip_path} -d /content/user_positives_raw
    # `unzip` may produce /content/user_positives_raw/positives/*.wav OR
    # /content/user_positives_raw/*.wav depending on how the user zipped it.
    raw = Path('/content/user_positives_raw')
    wavs = list(raw.rglob('*.wav'))
    print(f'  {len(wavs)} user wav(s) found in {zip_path.name}')
    for w in wavs:
        shutil.copy(w, USER_POSITIVES_DIR / w.name)
else:
    print(f'  (no zip at {zip_path}) — synthetic-only positives will be used.')

print(f'staged {len(list(USER_POSITIVES_DIR.glob("*.wav")))} user wav(s)')


## 3 — Download background audio (RIRs + AudioSet + FMA)

These get mixed into the positives during augmentation, teaching the model that
"Xiexie" said in a noisy living room is still "Xiexie". For full-strength training
you'd add more shards; for the hackathon T4 budget one each is plenty.


In [ ]:
# 3a) MIT room-impulse-responses (~10 MB).
rir_dir = Path('./mit_rirs')
rir_dir.mkdir(exist_ok=True)
if not list(rir_dir.glob('*.wav')):
    rirs = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True,
    )
    for row in tqdm(rirs, desc='MIT RIRs'):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(
            rir_dir / name, 16000,
            (row['audio']['array'] * 32767).astype(np.int16),
        )
print('rirs:', len(list(rir_dir.glob('*.wav'))))


In [ ]:
# 3b) AudioSet shard (~1.5 GB) — general background sounds.
shard = CONFIG['audioset_shard']
Path('audioset').mkdir(exist_ok=True)
tar_path = Path('audioset') / shard
if not tar_path.exists():
    !wget -q -O {tar_path} \
        https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{shard}
    !cd audioset && tar -xf {shard}

as16k = Path('./audioset_16k')
as16k.mkdir(exist_ok=True)
if len(list(as16k.glob('*.wav'))) < 100:
    flac_files = list(Path('audioset/audio').rglob('*.flac'))
    ds = datasets.Dataset.from_dict({'audio': [str(f) for f in flac_files]})
    ds = ds.cast_column('audio', datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds, desc='AudioSet -> 16k wav'):
        name = Path(row['audio']['path']).stem + '.wav'
        scipy.io.wavfile.write(
            as16k / name, 16000,
            (row['audio']['array'] * 32767).astype(np.int16),
        )
print('audioset_16k:', len(list(as16k.glob('*.wav'))))


In [ ]:
# 3c) Free Music Archive — N hours of music as additional negative background.
fma_dir = Path('./fma')
fma_dir.mkdir(exist_ok=True)
if len(list(fma_dir.glob('*.wav'))) < 50:
    fma = datasets.load_dataset(
        'rudraml/fma', name='small', split='train', streaming=True,
    )
    fma = iter(fma.cast_column('audio', datasets.Audio(sampling_rate=16000)))
    n_clips = CONFIG['fma_hours'] * 3600 // 30  # FMA-small clips are 30 s each.
    for _ in tqdm(range(n_clips), desc='FMA -> 16k wav'):
        try:
            row = next(fma)
        except StopIteration:
            break
        name = Path(row['audio']['path']).stem + '.wav'
        scipy.io.wavfile.write(
            fma_dir / name, 16000,
            (row['audio']['array'] * 32767).astype(np.int16),
        )
print('fma:', len(list(fma_dir.glob('*.wav'))))


## 4 — Pre-computed openWakeWord features

~2,000 h of negative examples already passed through the Google speech-embedding
backbone — saves us re-extracting features at training time. ~3 GB.


In [ ]:
if not Path('openwakeword_features_ACAV100M_2000_hrs_16bit.npy').exists():
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
if not Path('validation_set_features.npy').exists():
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print('feature files present:',
      Path('openwakeword_features_ACAV100M_2000_hrs_16bit.npy').exists(),
      Path('validation_set_features.npy').exists())


## 5 — Build the training YAML

openWakeWord's `train.py` is fully driven by a YAML file. We start from the
stock `examples/custom_model.yml` template and override the bits we care about.


In [ ]:
config = yaml.safe_load(open('openwakeword/examples/custom_model.yml').read())

config['model_name']         = CONFIG['model_name']
config['target_phrase']      = [CONFIG['wake_word']]
config['n_samples']          = CONFIG['n_samples_train']
config['n_samples_val']      = CONFIG['n_samples_val']
config['steps']              = CONFIG['steps']
config['layer_size']         = CONFIG['layer_size']
config['target_accuracy']    = CONFIG['target_accuracy']
config['target_recall']      = CONFIG['target_recall']
config['target_false_positives_per_hour'] = CONFIG['target_false_positives_per_hour']
config['max_negative_weight'] = CONFIG['max_negative_weight']

config['piper_sample_generator_path'] = './piper-sample-generator'
config['rir_paths']          = ['./mit_rirs']
config['background_paths']   = ['./audioset_16k', './fma']
config['background_paths_duplication_rate'] = [1, 1]
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['feature_data_files'] = {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}
config['output_dir']         = './my_custom_model'

Path('xiexie_model.yaml').write_text(yaml.safe_dump(config))
print('wrote xiexie_model.yaml')
print(yaml.safe_dump(config))


## 6 — Generate synthetic positives + adversarial negatives

Piper synthesises N variants of `"Xiexie"` using the multi-speaker libritts model
(varied speakers, length-scales, noise-scales) plus N phoneme-overlapping adversarial
phrases. ~10 min on T4.


In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py \
    --training_config xiexie_model.yaml --generate_clips


## 7 — Mix in the user's local recordings

We resample each user wav to 16 kHz mono int16 and copy `user_recording_duplication`
copies into `positive_train` (each augmented separately downstream, producing many
unique training examples from each real recording). User recordings carry far more
signal than synthetic clips because they include the actual mic, room, and voice the
runtime detector will see in the demo.


In [ ]:
import soundfile as sf
from scipy.signal import resample_poly

positive_train = Path('my_custom_model') / CONFIG['model_name'] / 'positive_train'
positive_train.mkdir(parents=True, exist_ok=True)

user_wavs = sorted(USER_POSITIVES_DIR.glob('*.wav'))
duplicates = CONFIG['user_recording_duplication']
TARGET_SR = 16000
TARGET_LEN_S = 1.5  # roughly the openWakeWord input window.
TARGET_LEN = int(TARGET_LEN_S * TARGET_SR)

added = 0
for w in user_wavs:
    audio, sr = sf.read(w, dtype='int16', always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1).astype(np.int16)
    if sr != TARGET_SR:
        audio = resample_poly(audio, TARGET_SR, sr).astype(np.int16)
    if len(audio) < TARGET_LEN:
        audio = np.pad(audio, (0, TARGET_LEN - len(audio)))
    elif len(audio) > TARGET_LEN:
        # centre-crop so we don't lose the wake-word at either edge.
        start = (len(audio) - TARGET_LEN) // 2
        audio = audio[start:start + TARGET_LEN]
    for k in range(duplicates):
        out = positive_train / f'user_{w.stem}_{k:02d}_{uuid.uuid4().hex[:6]}.wav'
        sf.write(out, audio, TARGET_SR, subtype='PCM_16')
        added += 1

print(f'mixed in {added} user-recording copies '
      f'from {len(user_wavs)} unique wavs (x{duplicates}).')
print('positive_train total:', len(list(positive_train.glob("*.wav"))))


## 8 — Augment + train

`--augment_clips` applies RIR convolutions and random noise from the AudioSet/FMA
shards, then runs the openWakeWord feature extractor on every clip — features are
memmapped during training so disk speed matters.

`--train_model` does the full auto-trainer (early stopping, checkpoint averaging,
cosine-decay LR, adaptive negative-weight schedule). Watch the log: it prints
validation accuracy / recall / false-pos-per-hour every few hundred steps and
stops when the targets in the YAML are met.


In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py \
    --training_config xiexie_model.yaml --augment_clips


In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py \
    --training_config xiexie_model.yaml --train_model


## 9 — Evaluation: confusion matrix on a held-out slice

We score the freshly-exported ONNX model on:

* `positive_test/` — synthetic Piper positives held back from training (recall).
* the AudioSet 16 kHz wavs — definitely-not-"Xiexie" audio (false-trigger rate).

These are not perfect proxies for production performance — for that you'd need real
recordings of "Xiexie" and a long noise corpus from the deployment environment — but
they give a quick sanity signal.


In [ ]:
from openwakeword.model import Model as OwwModel

onnx_path = Path('my_custom_model') / CONFIG['model_name'] / f"{CONFIG['model_name']}.onnx"
assert onnx_path.exists(), f'expected trained model at {onnx_path}'

model = OwwModel(wakeword_models=[str(onnx_path)], inference_framework='onnx')
model_key = list(model.models.keys())[0]
THRESHOLD = 0.5

def score_clip(path):
    sr, audio = scipy.io.wavfile.read(path)
    if sr != 16000:
        from scipy.signal import resample_poly
        audio = resample_poly(audio, 16000, sr).astype(np.int16)
    if audio.ndim > 1:
        audio = audio.mean(axis=1).astype(np.int16)
    model.reset()
    chunk = 1280  # 80 ms
    peak = 0.0
    for i in range(0, len(audio) - chunk + 1, chunk):
        s = model.predict(audio[i:i + chunk])[model_key]
        if s > peak:
            peak = s
    return peak

# Recall on synthetic positive_test.
pos_test = Path('my_custom_model') / CONFIG['model_name'] / 'positive_test'
pos_files = sorted(pos_test.glob('*.wav'))[:300]
pos_scores = [score_clip(p) for p in tqdm(pos_files, desc='positives')]
tp = sum(1 for s in pos_scores if s >= THRESHOLD)
fn = len(pos_scores) - tp
recall = tp / max(len(pos_scores), 1)

# False-trigger rate on AudioSet noise (definitely not 'Xiexie').
neg_files = sorted(Path('audioset_16k').glob('*.wav'))[:200]
neg_scores = [score_clip(p) for p in tqdm(neg_files, desc='negatives')]
fp = sum(1 for s in neg_scores if s >= THRESHOLD)
tn = len(neg_scores) - fp
false_trigger_rate = fp / max(len(neg_scores), 1)

print()
print('=' * 60)
print(f'  threshold        : {THRESHOLD}')
print(f'  positives tested : {len(pos_scores)}')
print(f'    TP={tp}  FN={fn}  -> recall = {recall:.3f}')
print(f'  negatives tested : {len(neg_scores)}')
print(f'    FP={fp}  TN={tn}  -> false-trigger = {false_trigger_rate:.3f}')
print(f'  positive score: min={min(pos_scores):.3f}  median={np.median(pos_scores):.3f}  max={max(pos_scores):.3f}')
print(f'  negative score: min={min(neg_scores):.3f}  median={np.median(neg_scores):.3f}  max={max(neg_scores):.3f}')
print('=' * 60)


## 10 — Export `xiexie.onnx` to Drive

Drops the model in `MyDrive/xiexie/xiexie.onnx`. Download it locally and place it at
`<repo>/models/xiexie.onnx`, then restart the backend — the runtime will pick it up
automatically (see `backend/xiexie/voice/wake.py`).


In [ ]:
drive_dir = Path(CONFIG['drive_output_dir'])
drive_dir.mkdir(parents=True, exist_ok=True)

drive_onnx = drive_dir / f"{CONFIG['model_name']}.onnx"
shutil.copy(onnx_path, drive_onnx)

# Optional: copy the YAML + a tiny eval summary so we know what produced this model.
shutil.copy('xiexie_model.yaml', drive_dir / 'xiexie_model.yaml')
(drive_dir / 'eval_summary.json').write_text(json.dumps({
    'threshold': THRESHOLD,
    'recall': recall,
    'false_trigger_rate': false_trigger_rate,
    'positives_tested': len(pos_scores),
    'negatives_tested': len(neg_scores),
    'config': CONFIG,
}, indent=2))

print('exported:')
print(' -', drive_onnx)
print(' -', drive_dir / 'xiexie_model.yaml')
print(' -', drive_dir / 'eval_summary.json')
